# Walkthrough Week 2 — Titanic Data Prep (แบบละเอียด)

ไฟล์นี้แตกโค้ดใน `student.py` ให้เป็นขั้นตอนสั้น ๆ พร้อม `print` ดูก่อน/หลัง

อ่านคู่กับ `สรุปการบ้าน_week2.md` (Obsidian)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df_raw = pd.read_csv("./titanic_to_student.csv", index_col=0)
print("shape:", df_raw.shape) # shape คือ จำนวนแถวและคอลัมน์
df_raw.head() # head คือ แสดงตัวอย่างของข้อมูล 5 แถวแรก (id -> 0-4) (index -> 1-5)

shape: (445, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1.0,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0,PC 17599,71.2833,C85,C
1,4,1.0,1.0,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0,113803,53.1000,C123,S
2,6,0.0,3.0,"Moran, Mr. James",male,NaN,0.0,0,330877,8.4583,NaN,Q
3,8,0.0,3.0,"Palsson, Master. Gosta Leonard",male,2.0,3.0,1,349909,21.0750,NaN,S
4,10,1.0,2.0,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1.0,0,237736,30.0708,NaN,C


## Q1 — นับแถว

แถว = จำนวนคน / observation

In [ ]:
df = df_raw.copy() # copy คือ สร้างตัวอย่างของข้อมูลเพื่อใช้ในการทำงาน

n_rows = len(df) # len คือ จำนวนแถว
n_rows_alt = df.shape[0] # shape คือ จำนวนแถวและคอลัมน์

print("len(df)     =", n_rows) # จำนวนแถว
print("df.shape[0] =", n_rows_alt) # จำนวนแถว
print("Answer Q1  =", n_rows)

## Q2.1 — ตัดคอลัมน์ที่ missing > 50%

เป้าหมาย: เก็บเฉพาะคอลัมน์ที่ว่าง **ไม่เกินครึ่ง**

ใน `student.py` เขียนบรรทัดเดียว:
```python
df = df.loc[:, df.isna().mean() <= 0.5]
```
ด้านล่างคือความหมายเดียวกันแบบแตกขั้น

In [49]:
df = df_raw.copy()

# ขั้น 1: ช่องไหนว่างบ้าง (True = ว่าง)
is_missing = df.isna() # isna() คือ ตรวจสอบว่าคอลัมน์ไหนมีค่า NaN ออกมาเป็น True หรือ False
print(is_missing[["Age", "Cabin"]].head())

     Age  Cabin
0  False  False
1  False  False
2   True   True
3  False   True
4  False   True


In [ ]:
# ขั้น 2: สัดส่วนว่างต่อคอลัมน์ (True=1, False=0 → mean = % ว่าง)
missing_ratio = is_missing.mean() # mean คือ ค่าเฉลี่ยของค่าที่มีค่า True หรือ False โดยค่าเฉลี่ยที่ออกมา เช่น 0.7 หมายความว่า False มี 70% ของค่าทั้งหมด
print(missing_ratio.sort_values(ascending=False))

Cabin          0.739326
Age            0.191011
Embarked       0.101124
Pclass         0.069663
Ticket         0.042697
SibSp          0.038202
Survived       0.029213
Name           0.026966
PassengerId    0.000000
Sex            0.000000
Parch          0.000000
Fare           0.000000
dtype: float64


In [26]:
# ขั้น 3: mask — คอลัมน์ไหนผ่านเกณฑ์ (<= 50% ว่าง)
mask = missing_ratio <= 0.5
print(mask)
print("\nคอลัมน์ที่ตัด:", list(missing_ratio.index[~mask]))

PassengerId     True
Survived        True
Pclass          True
Name            True
Sex             True
Age             True
SibSp           True
Parch           True
Ticket          True
Fare            True
Cabin          False
Embarked        True
dtype: bool

คอลัมน์ที่ตัด: ['Cabin']


In [ ]:
# ขั้น 4: เลือกคอลัมน์ตาม mask (ทุกแถว : , คอลัมน์ตาม mask)
print("ก่อน:", df.shape)
df = df.loc[:, mask] # loc ใช้สำหรับเลือกแถวตาม rows และคอลัมน์ตาม columns
print("หลัง:", df.shape)

# compact form (ผลเหมือนกัน):
# df = df.loc[:, df.isna().mean() <= 0.5]

ก่อน: (360, 11)
หลัง: (360, 11)


## Q2.2 — ตัด flat value > 70% (ยกเว้น Age, Fare)

flat = ค่าใดค่าหนึ่ง occupying ตารางเกือบทั้งคอลัมน์ (นับ NaN ด้วย → `dropna=False`)

In [ ]:
flat_cols = []
for col in df.columns:
    if col in ("Age", "Fare"):
        continue
    # สัดส่วนของค่าที่พบบ่อยสุด
    top_share = df[col].value_counts(normalize=True, dropna=False).max()
    print(f"{col:12} top_share={top_share:.4f}")
    if top_share > 0.7:
        flat_cols.append(col)

print("\nflat_cols ที่จะตัด:", flat_cols)
df = df.drop(columns=flat_cols)
print("หลังตัด flat:", df.shape)
print("Answer Q2 =", df.shape[1])

## Q3 — ลบแถวที่ Survived ว่าง

Target ห้ามเดา → ลบทั้งคน

In [ ]:
df = df_raw.copy()

n_missing_target = df["Survived"].isna().sum()
print("Survived ว่าง:", n_missing_target)

df_q3 = df.dropna(subset=["Survived"])
print("ก่อน:", len(df), "หลัง:", len(df_q3))
print("Answer Q3 =", len(df_q3))

## Q4 — clip Fare ด้วย IQR

ไม่ลบคน — ดึงค่าสุดโต่งกลับขอบ

In [ ]:
df = df_raw.copy()
fare = df["Fare"].copy()

q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"lower={lower}, upper={upper}")
print("n outliers:", ((fare < lower) | (fare > upper)).sum())
print("ตัวอย่างก่อน clip (สูงสุด 5 ค่า):", fare.nlargest(5).tolist())

fare_clipped = fare.clip(lower=lower, upper=upper)
# compact: fare.clip(lower=..., upper=...)

print("mean ก่อน:", round(fare.mean(), 2))
print("mean หลัง:", round(fare_clipped.mean(), 2))
print("Answer Q4 =", round(fare_clipped.mean(), 2))

## Q5 — impute ตัวเลขด้วย mean

เติมเฉพาะคอลัมน์ชนิดตัวเลข

In [ ]:
df = df_raw.copy()

print("Age ว่างก่อน:", df["Age"].isna().sum())
print("Age mean (ข้าม NaN):", df["Age"].mean())

num_cols = df.select_dtypes(include="number")
num_means = num_cols.mean()
print("\nmean ต่อคอลัมน์ตัวเลข (บางตัว):")
print(num_means[["Age", "Fare", "Survived"]])

df_filled = df.fillna(num_means)
# compact: df.fillna(df.select_dtypes(include="number").mean(), inplace=True)

print("\nAge ว่างหลัง:", df_filled["Age"].isna().sum())
print("Answer Q5 =", round(df_filled["Age"].mean(), 2))

## Q6 — mode แล้ว one-hot Embarked

ตาม slide: categorical missing → **mode** ก่อน แล้วค่อย dummy/one-hot

In [ ]:
df = df_raw.copy()

print("Embarked ก่อนเติม:")
print(df["Embarked"].value_counts(dropna=False))

mode_emb = df["Embarked"].mode()[0]
print("\nmode =", mode_emb)

df["Embarked"] = df["Embarked"].fillna(mode_emb)
print("\nEmbarked หลังเติม:")
print(df["Embarked"].value_counts(dropna=False))
print("NaN เหลือ:", df["Embarked"].isna().sum())

In [ ]:
# วิธีสั้น: get_dummies
dummy = pd.get_dummies(df, columns=["Embarked"])
print("คอลัมน์ใหม่:", [c for c in dummy.columns if c.startswith("Embarked_")])
print(dummy[["Embarked_C", "Embarked_Q", "Embarked_S"]].head())
print("mean Embarked_Q =", round(dummy["Embarked_Q"].mean(), 2))

In [ ]:
# วิธียาว (เหมือน student.py): OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore")
encoded = enc.fit_transform(df[["Embarked"]])
encoded_array = encoded.toarray()  # sparse → dense

col_names = [f"Embarked_{c}" for c in enc.categories_[0]]
dummy_ohe = pd.DataFrame(encoded_array, columns=col_names, index=df.index)

print("categories:", enc.categories_[0])
print("Answer Q6 =", round(dummy_ohe["Embarked_Q"].mean(), 2))

## Q7 — train/test 70:30 + stratify + seed 123

`main.py` จะ impute ตัวเลขให้ก่อนเรียก Q7 — ที่นี่ทำเหมือนกัน

In [ ]:
df = df_raw.copy()
df = df.fillna(df.select_dtypes(include="number").mean())

y = df["Survived"]
X = df.drop(columns=["Survived"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123, stratify=y
)

print("n all / train / test:", len(y), len(y_train), len(y_test))
print("prop survived==1 all  :", round((y == 1).mean(), 2))
print("prop survived==1 train:", round((y_train == 1).mean(), 2))
print("prop survived==1 test :", round((y_test == 1).mean(), 2))
print("Answer Q7 =", round((y_train == 1).mean(), 2))

## สรุปคำตอบ

| Q | Answer |
|---|--------|
| Q1 | 445 |
| Q2 | 10 |
| Q3 | 432 |
| Q4 | 26.27 |
| Q5 | 29.14 |
| Q6 | 0.06 |
| Q7 | 0.41 |